# Codetection permutation analysis

In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import switch_cwd_to_root

switch_cwd_to_root()

import os

import scanpy as sc

from spatial_tcr.codetection import test_gene_codetection_permutation
from spatial_tcr.tcr import get_tcr_genes
from spatial_tcr.utils import aggregate_gene_expression

In [ ]:
n_perm = 1000
data_dir = "data/xenium/processed"
out_dir = "results/revision"
os.makedirs(out_dir, exist_ok=True)

## Aggregated TRAV / TRBV / CD3 / TRBC (clonal clusters object)

In [ ]:
path = f"{data_dir}/08.1-kidney_tcr_clonal_clusters.h5ad"
adata = sc.read_h5ad(path)
adata.X = adata.layers["counts"].copy()
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)

ad_aggr = aggregate_gene_expression(
    adata,
    agg_dict={
        "TRAV": av_genes,
        "TRBV": bv_genes,
        "CD3": ["CD3D", "CD3E", "CD3G"],
        "TRBC": ["TRBC1", "TRBC2"],
    },
)
result = test_gene_codetection_permutation(
    ad_aggr,
    ["TRAV", "TRBV", "CD3", "TRAC", "TRBC"],
    "cell_type_l1",
    "cc",
    n_perm=n_perm,
)
result.to_csv(
    f"{out_dir}/codetection_permutation_result_aggr_genes_trv_cd3.csv",
    index=False,
)

## Per-gene TRAV / TRBV (infiltrate object)

In [ ]:
path = f"{data_dir}/05.2-kidney_tcr_infilrate.h5ad"
adata = sc.read_h5ad(path)
adata.X = adata.layers["counts"].copy()
av_genes, bv_genes, dv_genes, gv_genes, tv_genes = get_tcr_genes(adata)

In [ ]:
result = test_gene_codetection_permutation(
    adata, av_genes, "cell_type_l1", "cc", n_perm=n_perm
)
result.to_csv(f"{out_dir}/codetection_permutation_result_av_genes.csv", index=False)

In [ ]:
result = test_gene_codetection_permutation(
    adata, bv_genes, "cell_type_l1", "cc", n_perm=n_perm
)
result.to_csv(f"{out_dir}/codetection_permutation_result_bv_genes.csv", index=False)